In [19]:
import pandas as pd
import sys
import os

# Add parent directory to path so we can import src modules
sys.path.append(os.path.abspath('..'))

from sklearn.model_selection import train_test_split


In [20]:
df = pd.read_csv("../data/processed/engineered_dataset.csv")

# Split the data for evaluation
X = df.drop('attack_type', axis=1)
y = df['attack_type']
X_train_eval, X_test, y_train_eval, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

df.head()

,failed_connection_attempts,packets_sent_per_sec,packets_recv_per_sec,tcp_connection_count,context_switch_rate,unique_dst_port_count,top_process_cpu_pct,cpu_user_pct,load_avg_1m,attack_type,is_attack,packet_rate_total,packet_spike_ratio,connection_spike_ratio,cpu_spike_ratio,process_cpu_ratio,failure_rate,port_scan_intensity,load_per_connection
0,0.0,0.0,0.0,9.0,567.0,0,0.0,6.21,0.18,crypto,1,0.0,1.002004,1.000000,1.000000,0.000000,0.0,0.0,0.018000
1,0.0,0.0,0.0,9.0,803.0,0,0.0,1.50,0.41,crypto,1,0.0,1.002004,1.000000,0.389105,0.000000,0.0,0.0,0.041000
2,0.0,0.0,0.0,10.0,893.0,0,0.0,5.68,0.41,crypto,1,0.0,1.002004,1.071429,1.272591,0.000000,0.0,0.0,0.037273
3,0.0,0.0,0.0,14.0,404.0,0,26.0,4.99,0.41,crypto,1,0.0,1.002004,1.333333,1.085963,5.210410,0.0,0.0,0.027333
4,0.0,0.0,0.0,14.0,496.0,0,59.9,2.17,0.37,crypto,1,0.0,1.002004,1.250000,0.527981,27.603559,0.0,0.0,0.024667


In [21]:
from src.anomaly_model import AnomalyDetector
from src.classification_model import AttackClassifier

# Load anomaly model
anom = AnomalyDetector()
anom.load("../outputs/models/anomaly_model.pkl")

# Load classifier
clf = AttackClassifier()
clf.load("../outputs/models/classifier.pkl")

In [18]:
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    confusion_matrix,
    classification_report,
    precision_score,
    f1_score
)

from sklearn.ensemble import IsolationForest
import pandas as pd
import numpy as np

# =========================
# CONTAMINATION TEST VALUES
# =========================

contamination_values = [0.10, 0.15, 0.18, 0.20, 0.22, 0.25, 0.28]

results = []

# =========================
# LOOP THROUGH VALUES
# =========================

for contamination in contamination_values:

    print(f"\nTesting contamination = {contamination}")

    # =========================
    # TRAIN ISOLATION FOREST
    # =========================

    iso_forest = IsolationForest(
        contamination=contamination,
        random_state=42,
        n_estimators=100
    )

    # Train ONLY on normal traffic
    X_train_normal = X_train_eval[y_train_eval == 'none']

    iso_forest.fit(X_train_normal)

    # =========================
    # ANOMALY PREDICTION
    # =========================

    anomaly_raw = iso_forest.predict(X_test)

    # Convert:
    # 1 -> normal
    # -1 -> anomaly

    anomaly_pred = np.where(anomaly_raw == -1, 1, 0)

    # =========================
    # FINAL HYBRID PREDICTION
    # =========================

    final_predictions = []

    for i in range(len(X_test)):

        # If normal according to IF
        if anomaly_pred[i] == 0:
            final_predictions.append("none")

        # If anomaly -> use RF classifier
        else:
            rf_pred = clf.predict(X_test.iloc[i:i+1])[0]
            final_predictions.append(rf_pred)

    # =========================
    # EVALUATION
    # =========================

    accuracy = accuracy_score(y_test, final_predictions)

    # Attack recall
    attack_recall = recall_score(
        y_test != 'none',
        np.array(final_predictions) != 'none'
    )

    # DDoS recall
    ddos_mask = (y_test == 'ddos')

    ddos_recall = recall_score(
        y_test[ddos_mask],
        np.array(final_predictions)[ddos_mask],
        average='macro'
    )

    # False positives
    cm = confusion_matrix(y_test, final_predictions, labels=['none', 'crypto', 'ddos', 'portscan'])

    # normal predicted as attack
    false_positives = cm[0][1:].sum()

    # Precision
    precision = precision_score(
        y_test,
        final_predictions,
        average='weighted',
        zero_division=0
    )

    # F1 score
    f1 = f1_score(
        y_test,
        final_predictions,
        average='weighted'
    )

    # Store results
    results.append({
        'contamination': contamination,
        'accuracy': round(accuracy, 4),
        'attack_recall': round(attack_recall, 4),
        'ddos_recall': round(ddos_recall, 4),
        'false_positives': int(false_positives),
        'precision': round(precision, 4),
        'f1_score': round(f1, 4)
    })

# =========================
# FINAL RESULTS TABLE
# =========================

results_df = pd.DataFrame(results)

print("\n========== HYBRID SYSTEM COMPARISON ==========\n")
print(results_df)

# Optional sorting
results_df = results_df.sort_values(
    by=['attack_recall', 'accuracy'],
    ascending=False
)

print("\n========== SORTED RESULTS ==========\n")
print(results_df)


Testing contamination = 0.1

Testing contamination = 0.15

Testing contamination = 0.18

Testing contamination = 0.2

Testing contamination = 0.22

Testing contamination = 0.25

Testing contamination = 0.28

========== HYBRID SYSTEM COMPARISON ==========

   contamination  accuracy  attack_recall  ddos_recall  false_positives  \
0           0.10    0.9321         0.9971          1.0               75   
1           0.15    0.9009         0.9971          1.0              110   
2           0.18    0.8812         0.9971          1.0              132   
3           0.20    0.8643         0.9971          1.0              151   
4           0.22    0.8438         0.9971          1.0              174   
5           0.25    0.8241         0.9971          1.0              196   
6           0.28    0.7982         0.9971          1.0              225   

   precision  f1_score  
0     0.9605    0.9396  
1     0.9520    0.9147  
2     0.9479    0.8993  
3     0.9450    0.8861  
4     0.9419    0

In [ ]:
anomaly_preds, scores = anom.predict(X_test)

df['anomaly_pred'] = anomaly_preds

In [ ]:
attack_preds = clf.predict(X_test)

In [ ]:
final_predictions = [attack_preds[i] if anomaly_preds[i] == 1 else 'none' for i in range(len(X_test))]

In [9]:
from sklearn.metrics import classification_report, confusion_matrix

labels = sorted(set(y_test).union(set(final_predictions)))
cm = confusion_matrix(y_test, final_predictions, labels=labels)
cm_df = pd.DataFrame(
    cm,
    index=pd.Index(labels, name='Actual'),
    columns=pd.Index(labels, name='Predicted')
)

print("Confusion Matrix:\n")
display(cm_df)

print("\nClassification Report:\n")
print(classification_report(y_test, final_predictions, labels=labels))

Confusion Matrix:



Predicted,crypto,ddos,none,portscan
Actual,,,,
crypto,498,0,2,0
ddos,0,530,70,0
none,846,0,3054,0
portscan,0,0,0,600



Classification Report:

              precision    recall  f1-score   support

      crypto       0.37      1.00      0.54       500
        ddos       1.00      0.88      0.94       600
        none       0.98      0.78      0.87      3900
    portscan       1.00      1.00      1.00       600

    accuracy                           0.84      5600
   macro avg       0.84      0.92      0.84      5600
weighted avg       0.93      0.84      0.86      5600



In [10]:
print("Anomaly Detection Only:\n")
print(pd.Series(anomaly_preds).groupby(y_test).mean())

print("\nHybrid Final Output:\n")
print(pd.Series(final_predictions).groupby(y_test).value_counts())

Anomaly Detection Only:

attack_type
crypto      0.996000
ddos        0.883333
none        0.216923
portscan    1.000000
Name: anomaly_pred, dtype: float64

Hybrid Final Output:

attack_type  final_prediction
crypto       crypto               498
             none                   2
ddos         ddos                 530
             none                  70
none         none                3054
             crypto               846
portscan     portscan             600
Name: count, dtype: int64
